# Week-1 pilot: B1 and the agent on the dev set

Checks that the setup works on Kaggle, then runs `configs/b1_dev.yaml` and `configs/agent_dev.yaml`.
With only a handful of dev cases the scores mean little; the point is whether vLLM, tool calling
and constrained JSON work on a T4.

**Notebook settings:** Accelerator *GPU T4 x2*, Internet *on*.
If the GitHub repo is private, add a Kaggle secret named `GITHUB_TOKEN` (a read-only token).

Outputs go to `/kaggle/working/pilot_outputs/` (and `pilot_outputs.zip`).

In [ ]:
REPO_URL = "https://github.com/JagadishReddyK7/regrade-appeals-agent.git"
BRANCH = "main"
VLLM_SPEC = "vllm"  # pin the version that worked after the pilot, e.g. "vllm==X.Y.Z"
WORKDIR = "/kaggle/temp/regrade-appeals-agent"  # not saved with the notebook output
OUT = "/kaggle/working/pilot_outputs"

In [ ]:
# Clone. The token (if any) is removed from the git remote afterwards.
import os
import subprocess

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass  # public repo, or no secret configured
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(WORKDIR)
!git log -1 --oneline

In [ ]:
!pip install -q -e .
!pip install -q "{VLLM_SPEC}"
!python -c "import torch, vllm; print('torch', torch.__version__, 'vllm', vllm.__version__)"
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## Start vLLM
Downloads the model on first run (about 5.5 GB) and waits until the server answers.
If this fails, the next cell prints the end of the server log.

In [ ]:
!python -m llm.serve --config configs/b1_dev.yaml || tail -n 60 vllm.log

## Smoke test
Three checks: plain chat, a tool call with valid arguments, and schema-constrained JSON.
If one fails, the matching system will fail too: fix the server settings before trusting any run.

In [ ]:
!python -m llm.smoke --config configs/b1_dev.yaml

## Runs

In [ ]:
!python -m eval.run --config configs/b1_dev.yaml

In [ ]:
!python -m eval.run --config configs/agent_dev.yaml

## Save outputs and stop the server

In [ ]:
import shutil

os.makedirs(OUT, exist_ok=True)
shutil.copytree("runs", f"{OUT}/runs", dirs_exist_ok=True)
for name in ("vllm.log",):
    if os.path.exists(name):
        shutil.copy(name, OUT)
!pip freeze > {OUT}/pip_freeze.txt
!git rev-parse HEAD > {OUT}/commit.txt
shutil.make_archive("/kaggle/working/pilot_outputs", "zip", OUT)
!kill $(cat vllm.pid) 2>/dev/null; echo done